# Atividade P2 - Previsao de Churn de Clientes

Este notebook atualiza o trabalho da P1 mantendo o mesmo tema e dataset. As principais melhorias aplicadas foram:

- comparacao mais completa dos modelos;
- tratamento mais claro do desbalanceamento da classe `Churn`;
- interpretacao dos coeficientes da Regressao Logistica;
- discussao do trade-off entre `Precision` e `Recall`;
- escolha do modelo final e salvamento em `model/modelo_final.joblib`;
- geracao de arquivos usados pelo relatorio e pelo app Streamlit.


## 1. Importacao das bibliotecas


In [ ]:
from pathlib import Path
import json

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    roc_curve,
    auc,
    precision_recall_curve,
)


## 2. Configuracao de caminhos


In [ ]:
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data" / "dataset.csv").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = PROJECT_ROOT / "data" / "dataset.csv"
MODEL_DIR = PROJECT_ROOT / "model"
REPORTS_DIR = PROJECT_ROOT / "reports"
MODEL_DIR.mkdir(exist_ok=True)
REPORTS_DIR.mkdir(exist_ok=True)

DATA_PATH


## 3. Carregamento e inspecao inicial do dataset


In [ ]:
df = pd.read_csv(DATA_PATH)
print("Dimensoes iniciais:", df.shape)
print("Valores ausentes por coluna:")
print(df.isnull().sum())
df.head()


## 4. Limpeza e preparo inicial


In [ ]:
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
print("Ausentes em TotalCharges apos conversao:", df["TotalCharges"].isnull().sum())

df = df.dropna().copy()
df = df.drop(columns=["customerID"])
df["Churn"] = df["Churn"].map({"No": 0, "Yes": 1})

print("Dimensoes apos limpeza:", df.shape)
print(df["Churn"].value_counts())


## 5. Analise exploratoria e desbalanceamento


In [ ]:
churn_counts = df["Churn"].value_counts().rename(index={0: "Nao churn", 1: "Churn"})
churn_rate = df["Churn"].mean()
print(churn_counts)
print(f"Taxa de churn: {churn_rate:.2%}")

plt.figure(figsize=(5, 4))
sns.barplot(x=churn_counts.index, y=churn_counts.values, hue=churn_counts.index, palette="Set2", legend=False)
plt.title("Distribuicao da variavel alvo")
plt.ylabel("Quantidade")
plt.show()


In [ ]:
print("Churn por tipo de contrato:")
print(pd.crosstab(df["Contract"], df["Churn"], normalize="index"))

print("\nChurn por metodo de pagamento:")
print(pd.crosstab(df["PaymentMethod"], df["Churn"], normalize="index"))

print("\nChurn por tipo de internet:")
print(pd.crosstab(df["InternetService"], df["Churn"], normalize="index"))


## 6. Feature engineering: criacao da variavel tenure_group


In [ ]:
def tenure_group(value):
    if value <= 12:
        return "0-12"
    if value <= 24:
        return "13-24"
    if value <= 36:
        return "25-36"
    return "37+"


df["tenure_group"] = df["tenure"].apply(tenure_group).astype("category")
print(pd.crosstab(df["tenure_group"], df["Churn"], normalize="index"))


## 7. Separacao entre variaveis preditoras e alvo


In [ ]:
y = df["Churn"]
X = df.drop(columns=["Churn"])

num_cols = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
cat_cols = X.select_dtypes(include=["object", "string", "category"]).columns.tolist()

print("Colunas numericas:", num_cols)
print("Colunas categoricas:", cat_cols)


## 8. Pre-processamento com ColumnTransformer


In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(drop="first", handle_unknown="ignore"), cat_cols),
    ]
)


## 9. Divisao estratificada em treino, validacao e teste


In [ ]:
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.25, stratify=y_temp, random_state=42
)

print("Treino:", X_train.shape)
print("Validacao:", X_val.shape)
print("Teste:", X_test.shape)


## 10. Treinamento dos modelos


In [ ]:
models = {
    "Logistic Regression": Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("classifier", LogisticRegression(max_iter=2000, random_state=42, class_weight="balanced")),
        ]
    ),
    "Random Forest": Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("classifier", RandomForestClassifier(n_estimators=300, random_state=42, class_weight="balanced", n_jobs=-1)),
        ]
    ),
    "Gradient Boosting": Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("classifier", GradientBoostingClassifier(random_state=42)),
        ]
    ),
}

for name, model in models.items():
    model.fit(X_train, y_train)
    print(f"Modelo treinado: {name}")


## 11. Comparacao no conjunto de validacao


In [ ]:
def evaluate_predictions(y_true, y_pred, y_prob):
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1-Score": f1_score(y_true, y_pred, zero_division=0),
        "AUC-ROC": roc_auc_score(y_true, y_prob),
    }


validation_rows = []
preds = {}
probs = {}
for name, model in models.items():
    y_prob = model.predict_proba(X_val)[:, 1]
    y_pred = (y_prob >= 0.5).astype(int)
    preds[name] = y_pred
    probs[name] = y_prob
    validation_rows.append({"Modelo": name, **evaluate_predictions(y_val, y_pred, y_prob)})

validation_df = pd.DataFrame(validation_rows).sort_values(["AUC-ROC", "F1-Score"], ascending=False)
validation_df


## 12. Validacao cruzada estratificada


In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = {
    "Accuracy": "accuracy",
    "Precision": "precision",
    "Recall": "recall",
    "F1-Score": "f1",
    "AUC-ROC": "roc_auc",
}

cv_rows = []
for name, model in models.items():
    scores = cross_validate(model, X, y, cv=skf, scoring=scoring, n_jobs=-1)
    cv_rows.append({"Modelo": name, **{metric: scores[f"test_{metric}"].mean() for metric in scoring}})

cv_df = pd.DataFrame(cv_rows).sort_values(["AUC-ROC", "F1-Score"], ascending=False)
cv_df


## 13. Matrizes de confusao


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, (name, y_pred) in zip(axes, preds.items()):
    cm = confusion_matrix(y_val, y_pred)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=["Nao churn", "Churn"], yticklabels=["Nao churn", "Churn"], ax=ax)
    ax.set_title(name)
    ax.set_xlabel("Previsto")
    ax.set_ylabel("Real")
plt.tight_layout()
plt.show()


## 14. Curvas ROC/AUC


In [ ]:
plt.figure(figsize=(7, 5))
for name in models.keys():
    fpr, tpr, _ = roc_curve(y_val, probs[name])
    auc_score = auc(fpr, tpr)
    plt.plot(fpr, tpr, label=f"{name} (AUC = {auc_score:.4f})")
plt.plot([0, 1], [0, 1], linestyle="--", color="gray")
plt.xlabel("Taxa de falsos positivos")
plt.ylabel("Taxa de verdadeiros positivos")
plt.title("Curva ROC dos modelos")
plt.legend()
plt.show()


## 15. Feature importance do Gradient Boosting


In [ ]:
final_model_name = "Gradient Boosting"
final_model = models[final_model_name]

feature_names = final_model.named_steps["preprocessor"].get_feature_names_out()
importances = final_model.named_steps["classifier"].feature_importances_
importance_df = pd.DataFrame({"Feature": feature_names, "Importance": importances}).sort_values("Importance", ascending=False)
importance_df.head(10)


In [ ]:
plt.figure(figsize=(8, 5))
top10 = importance_df.head(10)
sns.barplot(data=top10, x="Importance", y="Feature", hue="Feature", palette="viridis", legend=False)
plt.title("Top 10 variaveis mais importantes - Gradient Boosting")
plt.show()


## 16. Interpretacao dos coeficientes da Regressao Logistica


In [ ]:
log_model = models["Logistic Regression"]
coef_df = pd.DataFrame({
    "Feature": log_model.named_steps["preprocessor"].get_feature_names_out(),
    "Coefficient": log_model.named_steps["classifier"].coef_[0],
})
coef_df["AbsCoefficient"] = coef_df["Coefficient"].abs()
coef_df = coef_df.sort_values("AbsCoefficient", ascending=False)
coef_df.head(10)


Coeficientes positivos aumentam a probabilidade estimada de churn. Coeficientes negativos reduzem essa probabilidade. Essa interpretacao ajuda a entender a direcao da relacao entre as variaveis e a saida do modelo.


## 17. Trade-off entre Precision e Recall


In [ ]:
precision, recall, thresholds = precision_recall_curve(y_val, probs[final_model_name])
f1 = np.divide(2 * precision * recall, precision + recall, out=np.zeros_like(precision), where=(precision + recall) != 0)
valid_thresholds = np.r_[thresholds, 1.0]
best_idx = int(np.nanargmax(f1))
best_threshold = float(valid_thresholds[best_idx])

print(f"Melhor threshold por F1 na validacao: {best_threshold:.3f}")
print(f"Precision: {precision[best_idx]:.4f}")
print(f"Recall: {recall[best_idx]:.4f}")
print(f"F1-Score: {f1[best_idx]:.4f}")

plt.figure(figsize=(7, 5))
plt.plot(recall, precision)
plt.scatter(recall[best_idx], precision[best_idx], color="red", label=f"Threshold {best_threshold:.3f}")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Curva Precision-Recall - Gradient Boosting")
plt.legend()
plt.show()


Em churn, aumentar o recall significa encontrar mais clientes com risco real de cancelamento. O custo e que a precision pode cair, gerando mais falsos positivos. Para a empresa, isso pode ser aceitavel quando a acao de retencao tem custo baixo em comparacao com a perda do cliente.


## 18. Avaliacao final no conjunto de teste


In [ ]:
y_test_prob = final_model.predict_proba(X_test)[:, 1]
y_test_pred_default = (y_test_prob >= 0.5).astype(int)
y_test_pred_tuned = (y_test_prob >= best_threshold).astype(int)

resultado_teste = pd.DataFrame([
    {"Cenario": "Threshold padrao 0.50", **evaluate_predictions(y_test, y_test_pred_default, y_test_prob)},
    {"Cenario": f"Threshold ajustado {best_threshold:.3f}", **evaluate_predictions(y_test, y_test_pred_tuned, y_test_prob)},
])
resultado_teste


In [ ]:
cm = confusion_matrix(y_test, y_test_pred_tuned)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=["Nao churn", "Churn"], yticklabels=["Nao churn", "Churn"])
plt.title("Matriz de confusao - teste com threshold ajustado")
plt.xlabel("Previsto")
plt.ylabel("Real")
plt.show()


## 19. Salvamento do modelo final


In [ ]:
joblib.dump(final_model, MODEL_DIR / "modelo_final.joblib")

metadata = {
    "model_name": final_model_name,
    "threshold": best_threshold,
    "input_columns": X.columns.tolist(),
    "numeric_columns": num_cols,
    "categorical_columns": cat_cols,
    "target": "Churn",
    "class_mapping": {"Nao churn": 0, "Churn": 1},
    "test_metrics_tuned_threshold": resultado_teste.iloc[1].to_dict(),
}

(MODEL_DIR / "metadata.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")
validation_df.to_csv(REPORTS_DIR / "comparacao_validacao.csv", index=False)
cv_df.to_csv(REPORTS_DIR / "validacao_cruzada.csv", index=False)
resultado_teste.to_csv(REPORTS_DIR / "resultado_teste.csv", index=False)
importance_df.head(20).to_csv(REPORTS_DIR / "feature_importance.csv", index=False)
coef_df.head(20).to_csv(REPORTS_DIR / "coeficientes_logistic_regression.csv", index=False)

print("Modelo salvo em:", MODEL_DIR / "modelo_final.joblib")
print("Metadados salvos em:", MODEL_DIR / "metadata.json")


## 20. Conclusao


O Gradient Boosting foi mantido como modelo final por apresentar bom desempenho geral em AUC-ROC. A melhoria principal da P2 foi tornar a decisao mais adequada ao problema de churn com ajuste de threshold, alem de documentar a comparacao entre modelos, interpretar a Regressao Logistica e salvar o modelo para uso no app Streamlit.
